# Falsehood control and direction comparison (Llama-3.2-3B)

**Part 1. What do the probes detect?** Build all four cases (C→C, C→I, I→C, I→I) for both of the authors' templates, extract activations once, train probes three ways, and check how often each probe calls a *false-ending but non-sycophantic* dialogue sycophantic. A falsehood detector says yes ~100% of the time; a caving detector says no.

**Part 2. Does a difference-in-means (DiM) direction steer where the probe direction didn't?** Same 16 heads, DiM direction instead of probe weights, steered with the authors' unchanged `inference_mha.py`; plus a Baez-style residual-stream DiM at one layer. Sign check at α = −5 and +5, judged with GPT-4o.

**Runtime:** GPU (T4) for sections 1, 4 and 5 (~30 min extraction, ~15 min per steering run). Sections 2, 3 and 6 run on CPU too. Every output goes to Drive, so you can switch runtimes between sections; just rerun the setup cells.

## 0. Setup (every new runtime)

In [ ]:
# Run once per runtime, then Runtime -> Restart session, then continue from the next cell.
!pip install -q transformers==4.57.6 pyvene==0.1.8 jsonlines

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
from huggingface_hub import login
login(userdata.get('HF_TOKEN'))

In [ ]:
REPO = '/content/sycophancy-in-MHA'
!test -d {REPO} || git clone -b falsehood-control https://github.com/<you>/sycophancy-in-MHA.git {REPO}
%cd {REPO}
!git pull -q && git log -1 --format='%h %s'

ROOT  = '/content/drive/MyDrive/syco'
FAITH = f'{ROOT}/llama/trained_probe_sycophancy/llama-3.2'   # faithful probes (already trained)
ACTS  = f'{ROOT}/falsehood/acts'                              # activations (written in section 1)
PROBES = f'{ROOT}/falsehood/probes'                           # probe results (section 2)
DIM   = f'{ROOT}/falsehood/dim'                               # DiM directions + steering outputs (sections 3-5)
!ls {FAITH} | grep -c linear_probe_   # should print 672

## 1. Build dialogues and extract activations (GPU, ~30 min)
About 5,200 dialogues (653 questions × 2 templates × 4 cases). Saved in chunks of 500; if the runtime disconnects, rerun this cell and it resumes. The script first checks that its subset reproduces the authors' `construct_data` exactly.

In [ ]:
!python falsehood/extract.py --model_id llama-3.2 --out_dir {ACTS}

In [ ]:
# The rendered chat template. Note the 'Today Date' line: Llama's template inserts the current date.
!cat {ACTS}/template_example.txt

## 2. Train and evaluate the probes (GPU or CPU, a few minutes)
- **A_original**: authors' data, labels and per-dialogue split (should roughly reproduce your faithful accuracies).
- **A_grouped**: same data and labels, split by question.
- **B**: all four cases, label 1 = sycophantic, split by question, class-balanced.
- **faithful**: your existing probes, evaluated as they are.

Key columns: `hit_rate` (sycophantic dialogues called sycophantic), `falsehood_ctrl_rate` (false-ending, non-sycophantic dialogues called sycophantic; ~1.0 for a falsehood detector), `syco_vs_ctrl_acc` (~0.5 for a falsehood detector).

In [ ]:
!python falsehood/probes.py --acts_dir {ACTS} --out_dir {PROBES} --faithful_dir {FAITH}

In [ ]:
import pandas as pd
from IPython.display import Image, display
cols = ['set', 'heads', 'val_best', 'val_last', 'hit_rate', 'falsehood_ctrl_rate', 'syco_vs_ctrl_acc',
        'challenged:C->I', 'challenged:I->I', 'hypothesis:I->I', 'hypothesis:C->I']
display(pd.read_csv(f'{PROBES}/summary.csv')[cols].round(3))
print(open(f'{PROBES}/sanity.json').read())
display(Image(f'{PROBES}/heatmaps.png'))

## 3. Difference-in-means directions (CPU ok)
Writes head DiM directions in the authors' probe-file format (with the faithful accuracy file, so the **same 16 heads** are steered), picks a residual layer by held-out separation, and prints how each direction lines up with the probe direction and with a pure falsehood direction.

In [ ]:
!python falsehood/dim.py --acts_dir {ACTS} --faithful_dir {FAITH} --out_dir {DIM}

## 4. Steer the same 16 heads with the DiM direction (GPU, ~15 min per run)
Points the authors' folders at the DiM files, then runs their unchanged `inference_mha.py`. Check that the printed "Top heads selected" list matches your faithful run.

In [ ]:
!rm -f probe/trained_probe_sycophancy predictions_sycophancy
!mkdir -p {DIM}/heads_dimA/predictions_sycophancy
!ln -s {DIM}/heads_dimA/trained_probe_sycophancy probe/trained_probe_sycophancy
!ln -s {DIM}/heads_dimA/predictions_sycophancy predictions_sycophancy
!ls -la probe/ | grep trained_probe; ls probe/trained_probe_sycophancy/llama-3.2 | grep -c std_mha   # 672

In [ ]:
!python inference_mha.py --model_id llama-3.2 --k_heads 16 --scale -5

In [ ]:
!python inference_mha.py --model_id llama-3.2 --k_heads 16 --scale 5

## 5. Steer the residual stream with the DiM direction (GPU, ~15 min per run)
Run the 3-question smoke test first and read the answers: if they are garbled, the scale is too large for this layer.

In [ ]:
!python falsehood/steer_residual.py --dim_file {DIM}/resid_dimA.pt --scale -5 --out_dir {DIM}/resid_predictions --limit 3
import glob; print(pd.read_csv(sorted(glob.glob(f'{DIM}/resid_predictions/*_first3.csv'))[-1])[['initial_answer', 'final_answer']].to_string())

In [ ]:
!python falsehood/steer_residual.py --dim_file {DIM}/resid_dimA.pt --scale -5 --out_dir {DIM}/resid_predictions

In [ ]:
!python falsehood/steer_residual.py --dim_file {DIM}/resid_dimA.pt --scale 5 --out_dir {DIM}/resid_predictions

## 6. Judge with GPT-4o (CPU) and compare

In [ ]:
import os, glob
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
files = sorted(glob.glob(f'{DIM}/heads_dimA/predictions_sycophancy/*_mha_linear.csv') + glob.glob(f'{DIM}/resid_predictions/*_dimA.csv'))
files = [f for f in files if '_judged_' not in f and '_first' not in f]
print('\n'.join(files))
for f in files:
    !python judge.py "{f}" --provider openai --votes 1

In [ ]:
cols = ['file', 'first_acc', 'second_acc', 'syco_rate', 'ci_low', 'ci_high', 'CC', 'CI', 'IC', 'II']
for label, p in [('Baseline', f'{ROOT}/llama/predictions_baseline'), ('Probe direction (faithful)', f'{ROOT}/llama/predictions_sycophancy'),
                 ('Random direction', f'{ROOT}/llama/predictions_random'), ('Head DiM', f'{DIM}/heads_dimA/predictions_sycophancy'),
                 ('Residual DiM', f'{DIM}/resid_predictions')]:
    s = pd.read_csv(f'{p}/judge_summary.csv')
    s = s[s.judge == 'gpt-4o-2024-08-06'][cols]
    if label == 'Probe direction (faithful)':
        s = s[s.file.str.contains('_16_(-5.0|5.0)_')]
    print(label); print(s.round(3).to_string(index=False)); print()